# Tahap 1 — Mengumpulkan dokumen berita ANTARA

Kita membangun sistem pencarian secara bertahap. Halaman utama nantinya langsung menampilkan kotak pencarian dan daftar hasil. Tahap ini menyiapkan dokumen yang akan dibaca Lucene.

Target pertama: menemukan tautan dari halaman Terkini, mengunjungi beberapa artikel, mengambil judul dan isi, lalu menyimpan dokumen ke JSON.

**Crawling** adalah proses menemukan dan mengikuti tautan halaman. **Scraping** adalah proses mengekstrak informasi dari HTML halaman yang sudah diambil.

Pada versi ini, crawling dibatasi pada satu halaman daftar dan lima artikel. Jalankan sel secara berurutan. Ini menggunakan situs ANTARA asli; isinya dapat berubah ketika dijalankan lagi.

## 1. Siapkan dependensi

Buka notebook ini di Jupyter atau VS Code. Pilih kernel Python dari lingkungan virtual proyek. Jalankan sel berikut jika dependensi belum tersedia. Instalasi cukup sekali untuk lingkungan tersebut.

In [1]:
%pip install requests beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2. Tentukan sumber dan batas pengambilan

Kita mulai dari halaman Terkini. Pemisahan konfigurasi di awal membuat sumber dan jumlah artikel mudah diperiksa. User-Agent memperkenalkan program ini sebagai crawler pembelajaran.

In [2]:
import json
import re
import time

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup


BASE_URL = "https://www.antaranews.com"

TARGET_ARTIKEL = 100
MAKS_HALAMAN = 10
JEDA_DETIK = 1.5

USER_AGENT = "PraktikumIR/1.0 (educational crawler)"

FOLDER_DATA = Path("data")
FOLDER_DATA.mkdir(exist_ok=True)

FILE_DATA = FOLDER_DATA / "dokumen_antara.json"

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

/usr/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


## 3. Siapkan pengambilan halaman

Program membaca robots.txt sekali, kemudian memeriksa alamat sebelum mengambil halaman. `requests.get` mengambil HTML, `raise_for_status()` memeriksa kegagalan HTTP, dan BeautifulSoup membaca struktur HTML.

Jika koneksi atau izin akses gagal, perbaiki penyebabnya sebelum melanjutkan. HTML halaman tidak memerlukan browser otomatis untuk struktur yang digunakan pada contoh ini.

In [3]:
response = session.get(BASE_URL + "/robots.txt", timeout=30)
response.raise_for_status()

robots = RobotFileParser()
robots.parse(response.text.splitlines())


class HentikanCrawl(RuntimeError):
    pass


def ambil_halaman(url):
    if not robots.can_fetch(USER_AGENT, url):
        raise HentikanCrawl(
            f"URL tidak diizinkan oleh robots.txt: {url}"
        )

    time.sleep(JEDA_DETIK)

    response = session.get(url, timeout=30)

    if response.status_code in (403, 429):
        raise HentikanCrawl(
            f"Server memberikan HTTP {response.status_code}. "
            "Proses dihentikan."
        )

    response.raise_for_status()
    response.encoding = "utf-8"

    return BeautifulSoup(response.text, "html.parser")

## 4. Crawling: temukan tautan berita

Pada HTML yang diperiksa tanggal 18 Oktober 2026, judul di halaman daftar berbentuk seperti ini:

~~~html
<h2 class="post_title post_title_medium">
  <a href="https://www.antaranews.com/berita/...">Judul berita</a>
</h2>
~~~

Selector **h2.post_title a[href]** berarti: cari tag h2 yang memiliki class post_title, lalu ambil tag a di dalamnya yang mempunyai atribut href.

Kita menyimpan hanya alamat artikel `/berita/`, sehingga video dan foto tidak ikut masuk. Fragmen dan parameter URL dibuang; alamat berulang dimasukkan satu kali.

In [4]:
def temukan_url_berita(soup, url_daftar):
    hasil = []

    for tautan in soup.select("h2.post_title a[href]"):
        url = urljoin(url_daftar, tautan["href"])
        bagian = urlparse(url)

        # Batasi ke situs utama ANTARA.
        if bagian.hostname not in {
            "www.antaranews.com",
            "antaranews.com",
        }:
            continue

        # Ambil halaman artikel berita saja.
        if not re.fullmatch(r"/berita/\d+/[^/]+/?", bagian.path):
            continue

        # Hilangkan parameter dan fragmen URL.
        url_bersih = BASE_URL + bagian.path.rstrip("/")

        if url_bersih not in hasil:
            hasil.append(url_bersih)

    return hasil


def ekstrak_artikel(soup, url):
    judul = soup.select_one("h1")
    isi = soup.select_one(".post-content")

    if judul is None or isi is None:
        raise ValueError("Elemen judul atau isi tidak ditemukan.")

    # Salin bagian isi agar pembersihan tidak mengubah soup asli.
    isi_bersih = BeautifulSoup(str(isi), "html.parser")

    selector_buang = (
        "script, style, noscript, iframe, ins, "
        ".baca-juga, .text-muted, .alert-info"
    )

    for elemen in isi_bersih.select(selector_buang):
        elemen.decompose()

    teks_judul = " ".join(
        judul.get_text(" ", strip=True).split()
    )
    teks_isi = " ".join(
        isi_bersih.get_text(" ", strip=True).split()
    )

    if not teks_judul or not teks_isi:
        raise ValueError("Judul atau isi kosong.")

    tanggal = soup.select_one(
        'meta[property="article:published_time"]'
    )
    nomor_berita = re.search(r"/berita/(\d+)/", url).group(1)

    return {
        "id": f"antara-{nomor_berita}",
        "title": teks_judul,
        "body": teks_isi,
        "url": url,
        "published_at": tanggal.get("content") if tanggal else None,
        "source": "ANTARA News",
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }


def simpan_dokumen(dokumen):
    FILE_DATA.write_text(
        json.dumps(dokumen, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

Periksa juga elemen aslinya. Bandingkan hasil berikut dengan Inspect pada browser. Variabel selector berasal dari struktur HTML yang diamati, bukan nama yang ditentukan sesuka kita.

## 5. Scraping: periksa satu artikel terlebih dahulu

Kita mengunjungi satu URL hasil crawling. Pada halaman artikel ANTARA yang diperiksa, judul berada pada h1 dan isi berada pada div dengan class post-content.

~~~html
<h1>Judul berita</h1>
<div class="wrap__article-detail-content post-content">
  <p>Isi berita...</p>
</div>
~~~

Class dapat lebih dari satu. Selector **.post-content** memilih elemen yang memiliki class post-content. Kita memeriksa jumlah kecocokannya sebelum mengekstrak teks.

In [9]:
dokumen = []
gagal = []

url_diperiksa = set()
id_tersimpan = set()

try:
    for halaman in range(1, MAKS_HALAMAN + 1):
        if len(dokumen) >= TARGET_ARTIKEL:
            break

        url_daftar = (
            f"{BASE_URL}/terkini"
            if halaman == 1
            else f"{BASE_URL}/terkini/{halaman}"
        )

        print(f"\nMembaca halaman daftar {halaman}")

        try:
            soup_daftar = ambil_halaman(url_daftar)
        except requests.RequestException as error:
            print("Gagal membaca daftar:", error)
            break

        urls = temukan_url_berita(soup_daftar, url_daftar)

        if not urls:
            print("Tidak ada URL artikel. Periksa selector halaman.")
            break

        urls_baru = [
            url for url in urls
            if url not in url_diperiksa
        ]

        if not urls_baru:
            print("Tidak ditemukan URL baru. Proses dihentikan.")
            break

        print(f"Ditemukan {len(urls_baru)} URL baru")

        for url in urls_baru:
            if len(dokumen) >= TARGET_ARTIKEL:
                break

            url_diperiksa.add(url)

            try:
                soup_artikel = ambil_halaman(url)
                artikel = ekstrak_artikel(soup_artikel, url)

                if artikel["id"] in id_tersimpan:
                    continue

                dokumen.append(artikel)
                id_tersimpan.add(artikel["id"])

                simpan_dokumen(dokumen)

                print(
                    f"[{len(dokumen)}/{TARGET_ARTIKEL}] "
                    f"{artikel['title']}"
                )

            except (requests.RequestException, ValueError) as error:
                gagal.append({
                    "url": url,
                    "error": str(error),
                })
                print("Gagal mengambil artikel:", error)

except HentikanCrawl as error:
    print(error)

finally:
    # Hindari menimpa file lama jika tidak ada hasil baru.
    if dokumen:
        simpan_dokumen(dokumen)

    (FOLDER_DATA / "gagal.json").write_text(
        json.dumps(gagal, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print(f"\nBerhasil: {len(dokumen)} artikel")
print(f"Gagal: {len(gagal)} artikel")
print("Lokasi hasil:", FILE_DATA.resolve())


Membaca halaman daftar 1
Ditemukan 13 URL baru
[1/100] Gol Julio Cesar bawa Persib Bandung menang 1-0 atas Persik Kediri
[2/100] PLN Batam: Penyelidikan penyebab gangguan listrik masih berlangsung
[3/100] Wamendiktisaintek: Pendidikan di perguruan tinggi harus lebih adaptif
[4/100] Polda Jabar bongkar pabrik obat keras produksi 8 juta butir per hari
[5/100] Prancis desak Israel tangkap penyerang diplomat Eropa
[6/100] Korsel janji terus upayakan untuk mewujudkan damai dengan Korut
[7/100] Filosofi Teduh gandeng Nowela rilis lagu Natal “Christmas Corazon”
[8/100] Survei PPI: Dedi Mulyadi teratas dalam simulasi elektabilitas cawapres
[9/100] Festival Baris Banten wadah bentuk karakter generasi muda nasionalis
[10/100] Erick Thohir berharap Liga Putri lahirkan talenta baru untuk timnas
[11/100] Kakorlantas instruksikan jajaran antisipasi kelangkaan BBM
[12/100] Tradisi Jepang warnai pembukaan Asian Para Games 2026
[13/100] PBSI: Ekosistem olahraga yang dikelola baik bisa tumbuhkan ekonom

## 6. Ambil field dokumen

Di dalam pembungkus isi juga terdapat iklan, script, tautan Baca juga, dan catatan penerbit. Elemen tersebut dikeluarkan dari **salinan pembungkus isi** sebelum teks diambil.

Judul, huruf besar-kecil, dan tanda baca tetap disimpan. Preprocessing untuk pencarian nanti diatur melalui Analyzer di Lucene.

Bentuk dokumen menggunakan field berbahasa Inggris agar konsisten dengan aplikasi pencarian: id, title, body, url, published_at, source, dan fetched_at. ID berasal dari nomor berita pada URL sehingga stabil ketika artikel yang sama diambil ulang.

In [6]:
def ekstrak_artikel(soup, url):
    judul = soup.select_one("h1")
    pembungkus = soup.select_one(".post-content")
    if judul is None or pembungkus is None:
        raise ValueError("Judul atau isi tidak ditemukan: " + url)

    isi_bersih = BeautifulSoup(str(pembungkus), "html.parser")
    selector_buang = "script, style, noscript, iframe, ins, .baca-juga, .text-muted, .alert-info"
    for elemen in isi_bersih.select(selector_buang):
        elemen.decompose()

    teks_judul = " ".join(judul.get_text(" ", strip=True).split())
    teks_isi = " ".join(isi_bersih.get_text(" ", strip=True).split())
    if not teks_judul or not teks_isi:
        raise ValueError("Judul atau isi kosong: " + url)

    tanggal = soup.select_one('meta[property="article:published_time"]')
    nomor_berita = re.search(r"/berita/(\d+)/", url).group(1)

    return {
        "id": "antara-" + nomor_berita,
        "title": teks_judul,
        "body": teks_isi,
        "url": url,
        "published_at": tanggal.get("content") if tanggal else None,
        "source": "ANTARA News",
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }


dokumen_pertama = ekstrak_artikel(soup_artikel_pertama, urls_berita[0])
print("ID:", dokumen_pertama["id"])
print("Judul:", dokumen_pertama["title"])
print("Tanggal:", dokumen_pertama["published_at"])
print("Jumlah karakter isi:", len(dokumen_pertama["body"]))
print("Cuplikan:", dokumen_pertama["body"][:160])

ID: antara-5794769
Judul: Wamenkomdigi ingatkan pers sajikan informasi yang jelas dan akurat
Tanggal: 2026-10-18T16:08:01+07:00
Jumlah karakter isi: 1847
Cuplikan: Pers juga untuk publik dan publik butuh pemberitaan yang sesuai dengan isinya Jakarta (ANTARA) - Wakil Menteri Komunikasi dan Digital (Wamenkomdigi) Angga Raka 


## 7. Terapkan pada beberapa URL

Artikel pertama sudah diambil dan diperiksa, sehingga kita gunakan kembali hasilnya. Program melanjutkan ke URL berikutnya sampai batas tercapai. Ada jeda antarpengambilan. Jika satu artikel gagal, alamat dan alasannya ditampilkan; kegagalan tidak disamarkan sebagai dokumen kosong.

In [7]:
dokumen = [dokumen_pertama]
gagal = []

for url in urls_berita[1:]:
    if len(dokumen) >= BATAS_ARTIKEL:
        break

    time.sleep(JEDA_DETIK)
    try:
        soup = ambil_halaman(url)
        artikel = ekstrak_artikel(soup, url)
        dokumen.append(artikel)
        print("OK", artikel["id"], artikel["title"])
    except (requests.RequestException, ValueError, RuntimeError) as error:
        gagal.append({"url": url, "error": str(error)})
        print("GAGAL", url, "|", error)

print("Berhasil:", len(dokumen), "| Gagal:", len(gagal))

OK antara-5794765 Perkuat Literasi Keuangan Generasi Muda, BRI Peduli Dukung Gerai CERDAS PeKA di Universitas Airlangga
OK antara-5794757 Survei PPI: 65,2 persen puas kinerja Presiden Prabowo
OK antara-5794753 Dua tahun Prabowo-Gibran, 10.070 Koperasi Desa Merah Putih beroperasi
OK antara-5794745 Analis dorong literasi hadapi jebakan pinjol saat daya beli lesu
Berhasil: 5 | Gagal: 0


## 8. Simpan dokumen sebagai JSON

File JSON menjadi penghubung tahap pengumpulan data dengan tahap indexing. Setiap elemen daftar merupakan satu artikel. File kegagalan disimpan terpisah agar masalah pengambilan dapat ditelusuri.

In [8]:
FOLDER_DATA.mkdir(parents=True, exist_ok=True)
path_dokumen = FOLDER_DATA / "dokumen_antara.json"
path_gagal = FOLDER_DATA / "gagal.json"

path_dokumen.write_text(
    json.dumps(dokumen, ensure_ascii=False, indent=2), encoding="utf-8"
)
path_gagal.write_text(
    json.dumps(gagal, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Dokumen tersimpan:", path_dokumen.resolve())

Dokumen tersimpan: /home/azani/Kuliah/semester_7/aslab/pi/apache_lucene/test_lucene/data/dokumen_antara.json


## 9. Periksa hasil sebelum masuk ke Lucene

Periksa bahwa file dapat dibaca kembali, ID dan URL unik, serta judul dan isi terisi. Pemeriksaan berikut memastikan struktur data siap dipakai tahap berikutnya; kualitas isi tetap perlu dilihat melalui cuplikan dan halaman sumber.

In [ ]:
hasil = json.loads(path_dokumen.read_text(encoding="utf-8"))

assert 0 < len(hasil) <= BATAS_ARTIKEL
assert len({d["id"] for d in hasil}) == len(hasil)
assert len({d["url"] for d in hasil}) == len(hasil)
assert all(d["title"].strip() and d["body"].strip() for d in hasil)

for d in hasil:
    print(d["id"], "|", len(d["body"]), "karakter |", d["title"])

print("Struktur dokumen siap untuk tahap indexing.")
session.close()

## Sesudah tahap ini

Kita akan memetakan title dan body menjadi TextField Lucene, serta id dan url sebagai field yang dapat diambil kembali. Setelah indexing dan pencarian terminal dipahami, endpoint pencarian dihubungkan dengan halaman utama yang langsung berisi kotak pencarian.

Pada tahap pertama ini, isi JSON masih teks asli hasil ekstraksi. Kita belum menjalankan BM25 atau membangun antarmuka web.

Sumber yang diperiksa: [Terkini ANTARA](https://www.antaranews.com/terkini) dan [robots.txt](https://www.antaranews.com/robots.txt), 18 Oktober 2026. Struktur halaman dan kebijakan situs dapat berubah. Gunakan konten sesuai ketentuan sumber.